In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [ ]:
!echo $SPARK_MASTER_ADDRESS

In [ ]:
spark = (
    SparkSession.builder
    .appName("Final_Project_Preprocessing")
    .config("spark.executor.memory", "20g")
    .config("spark.driver.memory", "20g")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

In [ ]:
YELP_REVIEW_PATH = "data/yelp_academic_dataset_review.json"
IMDB_PART01_PATH = "data/part-01.json"
KINDLE_CSV_PATH  = "data/preprocessed_kindle_review .csv"

In [ ]:
#Define schemas based on documentation infer schema was running out of memory for big JSON files, csv was fine.

imdb_schema = StructType([
    StructField("review_id", StringType(), True),      
    StructField("reviewer", StringType(), True),       
    StructField("movie", StringType(), True),           
    StructField("rating", DoubleType(), True),          
    StructField("review_summary", StringType(), True),  
    StructField("review_date", StringType(), True),     
    StructField("spoiler_tag", IntegerType(), True),    
    StructField("review_detail", StringType(), True),   
    StructField("helpful", ArrayType(IntegerType()), True)  
])

yelp_schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("business_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("text", StringType(), True),
    StructField("stars", DoubleType(), True),
    StructField("date", StringType(), True),
    StructField("useful", IntegerType(), True),
    StructField("funny", IntegerType(), True),
    StructField("cool", IntegerType(), True)
])

yelp_raw = spark.read.schema(yelp_schema).json(YELP_REVIEW_PATH)
imdb_raw = spark.read.schema(imdb_schema).json(IMDB_PART01_PATH)
kindle_raw = spark.read.csv(KINDLE_CSV_PATH, inferSchema=True, header=True)


# cut some out for speed
yelp_raw = yelp_raw.sample(withReplacement=False, fraction=0.2, seed=26)
imdb_raw = imdb_raw.sample(withReplacement=False, fraction=0.5, seed=26)

In [ ]:

print("=== YELP SAMPLE ===")
yelp_raw.show(3, truncate=50)

print("\n=== SKIPPING IMDB SHOW - transforming first ===")

print("\n=== KINDLE SAMPLE ===")
kindle_raw.show(3, truncate=50)

In [ ]:
def clean_review_text(col_name="review_text"):
    c = F.col(col_name)
    return F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(c, r"[\r\n\t]+", " "),
                r"\s+",
                " "
            ),
            r"[^\x20-\x7E]+",
            ""
        )
    )

def add_text_features(df, text_col="review_text"):
    return (
        df.withColumn("review_text", clean_review_text(text_col))
          .filter(F.col("review_text").isNotNull())
          .filter(F.col("review_text") != "")
          .withColumn("review_char_len", F.length("review_text"))
          .withColumn("review_word_count", F.size(F.split(F.col("review_text"), r"\s+")))
          .filter(F.col("review_word_count") >= 3)
    )

def normalize_rating_to_0_1(col_name, min_rating, max_rating):
    return (
        F.when(F.col(col_name).isNull(), None)
         .otherwise((F.col(col_name) - F.lit(min_rating)) / F.lit(max_rating - min_rating))
         .cast("double")
    )


# Process Yelp dataset (1-5 star scale)


print("Processing Yelp")

yelp_df = (
    yelp_raw.select(
        F.col("review_id").cast("string").alias("review_id"),
        F.lit("yelp").alias("source"),
        F.col("business_id").cast("string").alias("item_id"),
        F.lit(None).cast("string").alias("item_name"),
        F.col("user_id").cast("string").alias("reviewer_id"),
        F.col("text").cast("string").alias("review_text"),
        F.col("stars").cast("double").alias("rating_raw"),
        F.to_date("date").alias("review_date"),
        F.col("useful").cast("int").alias("helpful_votes"),
        F.col("funny").cast("int").alias("funny_votes"),
        F.col("cool").cast("int").alias("cool_votes")
    )
    .withColumn("rating_normalized", normalize_rating_to_0_1("rating_raw", 1.0, 5.0))
)

yelp_df = add_text_features(yelp_df, "review_text")

print("Yelp ready")

# Process IMDB dataset (1-10 scale)

print("Processing IMDB")

imdb_df = (
    imdb_raw.select(
        F.col("review_id").cast("string").alias("review_id"),
        F.lit("imdb").alias("source"),
        F.lit(None).cast("string").alias("item_id"),
        F.col("movie").cast("string").alias("item_name"),
        F.col("reviewer").cast("string").alias("reviewer_id"),
        F.col("review_detail").cast("string").alias("review_text"),
        F.col("rating").cast("double").alias("rating_raw"),
        F.to_date("review_date").alias("review_date"),
        F.when(F.col("helpful").isNotNull(), F.col("helpful")[0]).cast("int").alias("helpful_votes"),
        F.lit(None).cast("int").alias("funny_votes"),
        F.lit(None).cast("int").alias("cool_votes")
    )
    .withColumn("rating_normalized", normalize_rating_to_0_1("rating_raw", 1.0, 10.0))
)

imdb_df = add_text_features(imdb_df, "review_text")

print("IMDB ready")


# Process Kindle dataset (1-5 scale)


print("Processing Kindle")

kindle_df = (
    kindle_raw.select(
        F.concat(F.lit("kindle_"), F.col("_c0").cast("string")).alias("review_id"),
        F.lit("amazon_kindle").alias("source"),
        F.col("_c0").cast("string").alias("item_id"),
        F.lit(None).cast("string").alias("item_name"),
        F.lit(None).cast("string").alias("reviewer_id"),
        F.col("reviewText").cast("string").alias("review_text"),
        F.col("rating").cast("double").alias("rating_raw"),
        F.lit(None).cast("date").alias("review_date"),
        F.lit(None).cast("int").alias("helpful_votes"),
        F.lit(None).cast("int").alias("funny_votes"),
        F.lit(None).cast("int").alias("cool_votes")
    )
    .withColumn("rating_normalized", normalize_rating_to_0_1("rating_raw", 1.0, 5.0))
)

kindle_df = add_text_features(kindle_df, "review_text")

print("Kindle ready")


# Union all three datasets for final training

print("\nCombining all datasets...")

essential_columns = [
    "review_id",           # Tracking
    "source",              # Domain info
    "review_text",         # INPUT for BERT
    "rating_normalized",   # TARGET for BERT (0-1 scale)
    "rating_raw",          # Verification
    "review_word_count"    # Filtering
]

combined_df = (
    yelp_df.select(essential_columns)
    .unionByName(imdb_df.select(essential_columns), allowMissingColumns=True)
    .unionByName(kindle_df.select(essential_columns), allowMissingColumns=True)
    .filter(F.col("rating_normalized").isNotNull())
    .filter((F.col("rating_normalized") >= 0.0) & (F.col("rating_normalized") <= 1.0))
    .filter(F.col("review_word_count") >= 10)  # Filter very short reviews
    .filter(F.col("review_word_count") <= 512) # BERT max sequence length consideration
    .dropDuplicates(["source", "review_id"])
)

print("All datasets combined")

In [ ]:
combined_df = combined_df.withColumn(
    "rand", F.rand(seed=26)
).withColumn(
    "split",
    F.when(F.col("rand") < 0.7, "train")
     .when(F.col("rand") < 0.85, "val")
     .otherwise("test")
).drop("rand")
#Write to sqlite db


In [ ]:
output_base = os.path.abspath("parquet")

for split_name in ["train", "val", "test"]:
    
    print(f"\n{split_name.upper()} Split:")
    
    split_df = combined_df.filter(F.col("split") == split_name)
    
    # Write to Parquet with columnar compression
    output_path = f"{output_base}/{split_name}.parquet"
    
    split_df.write \
        .mode("overwrite") \
        .parquet(output_path)
